# Template train 1 model

Copy notebook này thành `<A|B|C>_<tên model>.ipynb` (ví dụ `B_lgb_baseline.ipynb`), rồi chỉ sửa 3 chỗ có `TODO`.

Mặc định đang là Logistic Regression để kiểm tra pipeline chạy đúng. Xóa output trước khi commit.

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from config import CAT_COLS, FEATURES, N_FOLDS, TARGET
from data import load_data
from utils import save_preds

MODEL_NAME = "logreg"  # TODO 1: tên model, dùng làm tên file oof_<tên>.npy

In [2]:
train, test = load_data()  # train có sẵn cột 'fold' và target 0/1
y = train[TARGET].values


def make_features(df):
    """TODO 2: feature engineering. Phải áp dụng giống nhau cho train và test."""
    X = df[FEATURES].copy()
    X["Arrival Delay in Minutes"] = X["Arrival Delay in Minutes"].fillna(X["Departure Delay in Minutes"])
    X = pd.get_dummies(X, columns=CAT_COLS, drop_first=True, dtype=float)
    return X


X, X_test = make_features(train), make_features(test)
X.shape, X_test.shape

((699635, 22), (299844, 22))

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def fit_predict(X_tr, y_tr, X_va, y_va, X_te):
    """TODO 3: thay bằng model của bạn. Trả về (xác suất trên X_va, xác suất trên X_te)."""
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(X_tr, y_tr)
    return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]

In [4]:
oof = np.zeros(len(train))
test_pred = np.zeros(len(test))

for k in range(N_FOLDS):
    tr, va = (train["fold"] != k).values, (train["fold"] == k).values
    oof[va], p_te = fit_predict(X[tr], y[tr], X[va], y[va], X_test)
    test_pred += p_te / N_FOLDS
    print(f"fold {k}: AUC = {roc_auc_score(y[va], oof[va]):.6f}")

print(f"CV AUC = {roc_auc_score(y, oof):.6f}")

fold 0: AUC = 0.922411
fold 1: AUC = 0.921415
fold 2: AUC = 0.921390
fold 3: AUC = 0.921276
fold 4: AUC = 0.921256
CV AUC = 0.921550


In [5]:
save_preds(MODEL_NAME, oof, test_pred, y)  # rồi ghi 1 dòng vào experiments.csv

Đã lưu oof_logreg.npy, test_logreg.npy | CV AUC = 0.921550
